# Phase-3D, L=4: the pocket and the single trivial↔trivial wall (positive octant)

Every trivial↔trivial cut we have (the x|z tails at h_y = 0–1.0 and the z|y lines at h_x = 0/0.2/0.5) sits on **one** first-order
surface: the boundary of the z-polarized (Higgs, ⟨B_p⟩≈1) region against the x/y-polarized regime (⟨B_p⟩≈0).
It starts on the pocket along **T** (topological | z-pol | x/y-pol meet), and ends on a rim **C** of critical endpoints.
There is no x|y wall: x- and y-polarized tilt smoothly into each other.

* **Data** — located L=4 boundary rows from `results/phase3d/summary.json` (the viewer's `located()` policy,
  `analysis/viewer/phase3d_viewer.html`: z|y lines at the level crossing); wall strength of each cut = max |⟨B_p⟩_up − ⟨B_p⟩_dn| over its up/dn overlap.
* **Wall corner, measured (2026-09-28)** — six h_z sweeps at fixed (h_x, h_y) ("bridge cuts", level crossings): (0.6, 1.3)
  0.375, (0.8, 1.2) 0.475, (1.0, 1.2) 0.625, (1.0, 1.4) 0.675 (+ (0.8, 1.0) 0.475 on the review-excluded h_y=1.0 cut, and
  (0.8, 1.4) weak / not located). Mean field − 0.06 predicted all of them within 0.06.
* **Assumed / extrapolated (flagged in §1)** — the rims: x-flank extrapolated past the last tail point; y-flank h_x=0 from
  the B_p fade, h_x=0.2/0.5 extrapolated; corner rays: B_p jump extrapolated linearly to 0. The corner of T (T_CORNER).

§3 = interactive 3D (plotly: drag to rotate, scroll to zoom, click legend entries to toggle). §4 = top view at fixed h_z.

In [ ]:
import json, os
import numpy as np
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from scipy.interpolate import RBFInterpolator
from ipywidgets import interact, FloatSlider

ROOT = (os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
        if os.getcwd().endswith(os.path.join("analysis", "notebooks")) else os.getcwd())
P3D = os.path.join(ROOT, "results", "phase3d")
FIGS = os.path.join(ROOT, "analysis", "figs")
plt.rcParams.update({"figure.dpi": 120, "font.size": 11, "axes.spines.top": False,
                     "axes.spines.right": False, "axes.grid": True, "grid.alpha": 0.3})
RED, BLUE, GREY = "#dc2626", "#2563eb", "#6b7280"
AX = {"hx": 0, "hy": 1, "hz": 2}

## §1 Config

In [ ]:
L = 4
TOPO_HZ = 0.3          # magnetic cuts at h_z <= TOPO_HZ bound the pocket (S2-confirmed); above: x|z tail
T_JUMP = 0.5           # nominal wall strength at T (there the wall merges with the pocket's first-order face)
X_RIM_DZ = 0.08        # x-flank rim: extend each tail past its last point by this much in h_z (B_p jump ~0 by h_z~0.9)
ZY_HX = (0.5, 0.2, 0.0)                     # y-flank lines (fixed h_x), ordered around the corner
RIM_Y = {0.0: (0, 1.78, .55), 0.2: (.2, 1.72, .54), 0.5: (.5, 1.56, .53)}   # h_x=0 from the B_p fade; others extrapolated
T_CORNER = (0.5, 1.05, 0.2)  # where the x-flank T (h_y=1.0: (0.5, 1.0, 0.206)) and y-flank T (h_x=0.5: (0.5, 1.075, 0.2)) meet
BRIDGE_RAYS = [((0.8, 1.2), (1.0, 1.2)),    # wall corner, MEASURED (bridge h_z sweeps 2026-09-27/28, level crossings): rays from
               ((0.6, 1.3), (1.0, 1.4))]    # T_CORNER through located bridge cuts (h_x, h_y), inner -> outer; each ray's rim =
                                            # its B_p jump extrapolated linearly to 0 past the outer point (EXTRAPOLATED)
HZ_SLICES = (0.4, 0.5, 0.7, 0.85)
SHOW_MF = True         # faint mean-field (product state, L=4 OBC stabilizer weights) wall in the top view

## §2 Data: located boundary rows and measured wall strength

In [ ]:
def located(cut, L):
    # (h_c, err) of one cut at size L, or None -- mirrors located() in analysis/viewer/phase3d_viewer.html
    L = str(L)
    if cut.get("excluded"):
        return None
    get = lambda key: (cut.get(key) or {}).get(L)
    ok = lambda r: (r["h_c"], r["err"]) if r and r.get("ok") else None
    jump, hc, cross, loop, level = get("jump"), get("hc"), get("crossing"), get("loop"), get("level")
    if cut.get("jump_primary"):
        return ok(jump)
    efo = cut["kind"] == "electric" and bool(cut.get("first_order"))
    if not efo and hc and hc.get("h_c") is not None:
        return hc["h_c"], hc["err"]
    if efo:        # z|y wall (09-27): lower-energy branch switches up -> dn = level crossing; x-pol -> z-pol: M_z jump
        if cut.get("loop_primary"):    # level crossing masked by a stuck up start (review 09-28): loop centre
            return ok(loop)
        return ok(level) if level else ok(jump)
    crossed = cross and cross.get("h_c") is not None and not cross.get("merged")
    hz = cut["fixed"].get("hz")
    if cut["kind"] == "ycut" or (cut["kind"] != "electric" and (hz is None or hz > TOPO_HZ)):
        return ok(jump) or ((cross["h_c"], cross["err"]) if crossed else None) or ok(loop)
    return (cross["h_c"], cross["err"]) if crossed else None


def cut_jump(cut, L):
    # Wall strength: max |B_p(up) - B_p(dn)| over the fields where both branches landed (nan if no overlap)
    br = {}
    for p in cut["points"]:
        if p["L"] == L and not p.get("diverged") and p.get("B_p") is not None:
            br.setdefault(p["branch"], {})[round(p["h"], 4)] = p["B_p"]
    common = set(br.get("up", {})) & set(br.get("dn", {}))
    return max((abs(br["up"][h] - br["dn"][h]) for h in common), default=np.nan)


SUMMARY = json.load(open(os.path.join(P3D, "summary.json")))
CUTS = {(plane["hy"], cut["id"]): cut for plane in SUMMARY["planes"] for cut in plane["cuts"]}
rows = []
for (hy, cid), cut in CUTS.items():
    kind = {"electric": "electric", "ycut": "ycut"}.get(cut["kind"], "magnetic")
    fixed = cut["fixed"] if kind == "ycut" else {**cut["fixed"], "hy": hy}
    loc = located(cut, L)
    if loc:
        pt = {**fixed, cut["sweep"]: loc[0]}
        topo = {"electric": not cut.get("first_order"), "ycut": bool(cut.get("topo")),
                "magnetic": not cut.get("jump_primary") and fixed.get("hz", 1) <= TOPO_HZ}[kind]
        rows.append(dict(hx=pt["hx"], hy=pt["hy"], hz=pt["hz"], kind=kind, topo=topo, cut=cid,
                         order=1 if kind != "electric" or cut.get("first_order") else 2, jump=cut_jump(cut, L)))
print(f"{len(rows)} located L={L} rows: {sum(r['topo'] for r in rows)} pocket, {sum(not r['topo'] for r in rows)} trivial<->trivial")

In [ ]:
xyz = lambda rs: np.array([[r["hx"], r["hy"], r["hz"]] for r in rs], float)

def monotone(P, J):
    # drop points that do not climb in h_z (same-h_z duplicates / zig-zag between electric and magnetic cuts);
    # cuts without an up/dn overlap (cold 7-point sweeps) get their jump interpolated along the line
    keep = np.r_[True, np.diff(P[:, 2]) > 0.02]
    P, J = P[keep], J[keep].astype(float)
    ok = ~np.isnan(J)
    return P, np.interp(np.arange(len(J)), np.flatnonzero(ok), J[ok])

def x_flank(hy):
    # x|z tail of one h_y plane: pocket corner T, the plane's trivial rows by h_z, extrapolated rim
    plane = [r for r in rows if r["kind"] != "ycut" and r["hy"] == hy]
    elec = sorted((r for r in plane if r["kind"] == "electric" and r["topo"]), key=lambda r: r["hx"])
    mag = sorted((r for r in plane if r["kind"] == "magnetic" and r["topo"]), key=lambda r: -r["hz"])
    tail = sorted((r for r in plane if not r["topo"]), key=lambda r: r["hz"])
    if not tail or not (elec or mag):
        return None
    if elec and mag and any(r["kind"] == "electric" for r in tail):   # first-order h_z sweep cuts the corner
        T = {k: (elec[-1][k] + mag[0][k]) / 2 for k in AX}
    else:
        T = elec[-1] if elec else mag[0]
    P, J = monotone(xyz([T] + tail), np.array([T_JUMP] + [r["jump"] for r in tail]))
    a, b = P[-2], P[-1]
    return np.vstack([P, b + (b - a) * X_RIM_DZ / (b[2] - a[2])]), np.r_[J, 0.0]

def y_flank(hx):
    # z|y line at fixed h_x: roof tip T, trivial y-cuts and first-order h_z sweeps by h_y, rim
    ycuts = [r for r in rows if r["kind"] == "ycut" and r["hx"] == hx]
    tip = sorted((r for r in ycuts if r["topo"]), key=lambda r: -r["hz"])[:1]
    pts = [r for r in ycuts if not r["topo"]] + [r for r in rows if r["kind"] == "electric" and not r["topo"] and r["hx"] == hx]
    pts = sorted(pts, key=lambda r: r["hy"])
    P, J = monotone(xyz(tip + pts), np.array([T_JUMP] + [r["jump"] for r in pts]))
    return np.vstack([P, RIM_Y[hx]]), np.r_[J, 0.0]

x_lines = [l for l in (x_flank(hy) for hy in sorted({r["hy"] for r in rows if r["kind"] != "ycut"})) if l]
def bridge_ray(cells):
    # T_CORNER -> located bridge wall points -> rim where the linearly extrapolated B_p jump reaches 0
    pts = [next(r for r in rows if r["kind"] == "electric" and not r["topo"] and (r["hx"], r["hy"]) == c) for c in cells]
    P, J = xyz(pts), np.array([r["jump"] for r in pts], float)
    a, b = P[-2], P[-1]
    return np.vstack([T_CORNER, P, b + J[-1] / (J[-2] - J[-1]) * (b - a)]), np.r_[T_JUMP, J, 0.0]

corner = [bridge_ray(c) for c in BRIDGE_RAYS]
LINES = x_lines + corner + [y_flank(hx) for hx in ZY_HX]           # ordered around the wall: x-flank -> corner -> y-flank
KIND = ["data"] * len(x_lines) + ["data"] * len(corner) + ["data"] * len(ZY_HX)   # corner measured since 2026-09-28
for (P, J), k in zip(LINES, KIND):
    print(k, np.round(P[0], 3), "->", np.round(P[-1], 3), " jumps", np.round(J, 2))

def slice_runs(hz):
    # runs of consecutive wall lines that reach height hz -> [(points (k,2), kinds (k,))]
    runs, cur = [], []
    for (P, _), k in zip(LINES, KIND):
        if P[0, 2] <= hz <= P[-1, 2]:
            i = max(np.searchsorted(P[:, 2], hz), 1)
            a, b = P[i - 1], P[i]
            cur.append(((a + (hz - a[2]) / (b[2] - a[2]) * (b - a))[:2], k))
        elif cur:
            runs.append(cur); cur = []
    runs += [cur] if cur else []
    return [(np.array([p for p, _ in r]), np.array([k for _, k in r])) for r in runs]

## §3 Interactive 3D (drag to rotate · scroll to zoom · click legend to toggle)

In [ ]:
def sphere_rbf(P, values, smoothing):
    # thin-plate RBF over the directions of P, mirrored into all octants (star-shaped pocket)
    U = P / np.linalg.norm(P, axis=1)[:, None]
    S = np.array([(a, b, c) for a in (1, -1) for b in (1, -1) for c in (1, -1)])
    U8, keep = np.unique(np.round(np.concatenate([U * s for s in S]), 9), axis=0, return_index=True)
    return RBFInterpolator(U8, np.tile(values, len(S))[keep], kernel="thin_plate_spline", smoothing=smoothing)

def pocket_surface(n=60, smoothing=3e-3):
    topo = [r for r in rows if r["topo"]]
    P = xyz(topo)
    th, ph = np.meshgrid(np.linspace(0, np.pi / 2, n), np.linspace(0, np.pi / 2, n))
    U = np.stack([np.sin(th) * np.cos(ph), np.sin(th) * np.sin(ph), np.cos(th)], -1).reshape(-1, 3)
    R = sphere_rbf(P, np.linalg.norm(P, axis=1), smoothing)(U)
    second = sphere_rbf(P, np.array([r["order"] == 2 for r in topo], float), 1e-2)(U)
    X, Y, Z = (U * R[:, None]).T.reshape(3, n, n)
    return X, Y, Z, np.clip(second.reshape(n, n), 0, 1)

def resample(P, J, n=30):
    s = np.r_[0, np.cumsum(np.linalg.norm(np.diff(P, axis=0), axis=1))]
    t = np.linspace(0, s[-1], n)
    return np.stack([np.interp(t, s, P[:, k]) for k in range(3)], -1), np.interp(t, s, J)

def wall_mesh(pairs, name, colorscale, n=30):
    # Mesh3d spanning consecutive polylines; vertex intensity = B_p jump
    V, I, F = [], [], []
    for (Pa, Ja), (Pb, Jb) in pairs:
        (a, ja), (b, jb) = resample(Pa, Ja, n), resample(Pb, Jb, n)
        o = sum(len(v) for v in V)
        V += [a, b]; I += [ja, jb]
        for k in range(n - 1):
            F += [(o + k, o + k + 1, o + n + k), (o + k + 1, o + n + k + 1, o + n + k)]
    V, I, F = np.vstack(V), np.concatenate(I), np.array(F)
    return go.Mesh3d(x=V[:, 0], y=V[:, 1], z=V[:, 2], i=F[:, 0], j=F[:, 1], k=F[:, 2], intensity=I,
                     cmin=0, cmax=0.5, colorscale=colorscale, opacity=0.75, name=name, showlegend=True,
                     showscale=name.startswith("wall"), colorbar=dict(title="B_p jump", len=0.5, x=0.9),
                     hoverinfo="skip")

X, Y, Z, second = pocket_surface()
fig = go.Figure()
fig.add_trace(go.Surface(x=X, y=Y, z=Z, surfacecolor=second, cmin=0, cmax=1, showscale=False, opacity=0.5,
                         colorscale=[[0, "#f4b4b4"], [1, "#b4c8f4"]], name="pocket (red 1st, blue 2nd order)",
                         showlegend=True, hoverinfo="skip"))
pairs = list(zip(LINES, LINES[1:]))
data_pairs = [p for p, k0, k1 in zip(pairs, KIND, KIND[1:]) if k0 == k1 == "data"]
mf_pairs = [p for p, k0, k1 in zip(pairs, KIND, KIND[1:]) if "mf" in (k0, k1)]
fig.add_trace(wall_mesh(data_pairs, "wall z-pol | x/y-pol (data)", [[0, "#fde2e2"], [1, "#b91c1c"]]))
if mf_pairs:
    fig.add_trace(wall_mesh(mf_pairs, "corner (unmapped, MF-guided)", [[0, "#e5e7eb"], [1, "#4b5563"]]))
T = np.array([P[0] for P, _ in LINES]); C = np.array([P[-1] for P, _ in LINES])
fig.add_trace(go.Scatter3d(x=T[:, 0], y=T[:, 1], z=T[:, 2], mode="lines", line=dict(color="black", width=6, dash="dash"),
                           name="T: wall meets pocket (3 phases)"))
fig.add_trace(go.Scatter3d(x=C[:, 0], y=C[:, 1], z=C[:, 2], mode="lines", line=dict(color="#7f1d1d", width=6, dash="dot"),
                           name="C: rim (critical endpoints, estimated)"))
first = True
for (P, J), k in zip(LINES, KIND):
    if k != "data":
        continue
    txt = [f"h=({x:.3f}, {y:.3f}, {z:.3f})<br>B_p jump {j:.2f}" for (x, y, z), j in zip(P[1:-1], J[1:-1])]
    fig.add_trace(go.Scatter3d(x=P[:-1, 0], y=P[:-1, 1], z=P[:-1, 2], mode="lines", line=dict(color=RED, width=3),
                               legendgroup="pts", showlegend=False, hoverinfo="skip"))
    fig.add_trace(go.Scatter3d(x=P[1:-1, 0], y=P[1:-1, 1], z=P[1:-1, 2], mode="markers", text=txt, hoverinfo="text",
                               marker=dict(size=3.5, color=RED), legendgroup="pts", showlegend=first,
                               name="located L=4 wall points"))
    first = False
topo_pts = xyz([r for r in rows if r["topo"]])
fig.add_trace(go.Scatter3d(x=topo_pts[:, 0], y=topo_pts[:, 1], z=topo_pts[:, 2], mode="markers",
                           marker=dict(size=2.5, color="#374151"), name="located pocket points", visible="legendonly"))
for hz in HZ_SLICES:        # the top-view cuts of §4 as 3D curves (off by default: click the legend)
    for n_run, (Q, _) in enumerate(slice_runs(hz)):
        fig.add_trace(go.Scatter3d(x=Q[:, 0], y=Q[:, 1], z=np.full(len(Q), hz), mode="lines+markers",
                                   line=dict(color="#7c3aed", width=4), marker=dict(size=2), legendgroup=f"s{hz}",
                                   name=f"cut at h_z={hz}", showlegend=n_run == 0, visible="legendonly"))
fig.update_layout(scene=dict(xaxis=dict(title="h_x", range=[0, 1.6]), yaxis=dict(title="h_y", range=[0, 1.9]),
                             zaxis=dict(title="h_z", range=[0, 1.0]), aspectmode="manual",
                             aspectratio=dict(x=1.6, y=1.9, z=1.1), camera=dict(eye=dict(x=1.7, y=1.3, z=0.9))),
                  scene_annotations=[dict(x=1.5, y=0.1, z=0.05, text="x-pol", showarrow=False),
                                     dict(x=0.1, y=1.8, z=0.05, text="y-pol", showarrow=False),
                                     dict(x=0.15, y=0.2, z=0.85, text="z-pol", showarrow=False),
                                     dict(x=1.4, y=1.6, z=0.1, text="x/y canted: no wall", showarrow=False)],
                  legend=dict(x=0, y=1), height=720, margin=dict(l=0, r=0, t=30, b=0),
                  title=f"L={L}: pocket + one trivial wall (positive octant)")
fig.show()
# fig.write_html(os.path.join(FIGS, "phase3d_trivial_wall_L4.html"))

## §4 Top view: the wall at fixed h_z (positive quadrant)

Each colour = one horizontal cut. Inside the curve (toward the origin): z-pol; outside: x/y-pol. Filled markers lie on
measured wall lines (interpolated between located points); the corner is measured since 2026-09-28 (bridge cuts). A curve that
stops short of an axis = the wall has already ended there (above the rim). Faint = mean field with L=4 OBC weights.

In [ ]:
def mf_slice(hz, g=np.arange(0, 2.0, 0.02), M=121):
    # product-state mean field, L=4 OBC weights: jump midpoints (hx, hy, |dn|) of the minimiser on h_z = hz
    th, ph = np.meshgrid(np.linspace(0, np.pi / 2, M), np.linspace(0, np.pi / 2, M), indexing="ij")
    n = np.stack([np.sin(th) * np.cos(ph), np.sin(th) * np.sin(ph), np.cos(th)], -1).reshape(-1, 3)
    nx, nz = n[:, 0], n[:, 2]
    e0 = -(8 * nx**3 + 24 * nx**4 + 24 * nx**5 + 8 * nx**6) / 144 - 0.75 * nz**4   # 3/4/5/6-leg stars, 108 plaquettes, 144 edges
    A, B = np.meshgrid(g, g, indexing="ij")
    H = np.stack([A, B, np.full_like(A, hz)], -1).reshape(-1, 3)
    N = n[np.concatenate([(e0[None] - H[s:s + 1024] @ n.T).argmin(1) for s in range(0, len(H), 1024)])]
    N = N.reshape(A.shape + (3,))
    out = []
    for d in (0, 1):
        dn = np.linalg.norm(np.diff(N, axis=d), axis=-1)
        i, j = np.nonzero(dn > 0.08)
        i2, j2 = (i + 1, j) if d == 0 else (i, j + 1)
        out.append(np.column_stack([(A[i, j] + A[i2, j2]) / 2, (B[i, j] + B[i2, j2]) / 2, dn[i, j]]))
    return np.vstack(out)

footprint = sorted([(r["hx"], r["hy"]) for r in rows if r["topo"] and r["hz"] == 0], key=lambda p: np.arctan2(p[1], p[0]))
SLICE_COL = dict(zip(HZ_SLICES, plt.cm.plasma(np.linspace(0, 0.85, len(HZ_SLICES)))))

def top_view(hzs, ax=None):
    ax = ax or plt.subplots(figsize=(6.4, 6.4))[1]
    ax.fill(*np.array([(0, 0)] + footprint + [(0, 0)]).T, color="#ebe8f3", lw=0, zorder=0)
    ax.text(0.3, 0.45, "pocket\n($h_z$=0 footprint)", ha="center", fontsize=8, color="0.4")
    for hz in hzs:
        col = SLICE_COL.get(hz, "k")
        if SHOW_MF:
            W = mf_slice(hz)
            ax.scatter(W[:, 0], W[:, 1], s=3, color=col, alpha=np.clip(W[:, 2], 0.05, 0.35), lw=0, zorder=1)
        for n_run, (P, K) in enumerate(slice_runs(hz)):
            ax.plot(*P.T, "-", color=col, lw=1.4, zorder=2, label=f"$h_z$={hz}" if n_run == 0 else None)
            ax.plot(*P[K == "data"].T, "o", color=col, ms=4, zorder=3)
            ax.plot(*P[K == "mf"].T, "o", mfc="w", mec=col, ms=4.5, zorder=3)
    for x, y, t in ((1.75, 0.15, "x-pol"), (0.15, 1.9, "y-pol"), (1.5, 1.6, "x/y canted\n(no wall)"), (0.15, 1.0, "z-pol")):
        ax.text(x, y, t, ha="center", fontsize=9, style="italic", color="0.3")
    ax.set(xlim=(0, 2.0), ylim=(0, 2.05), aspect="equal", xlabel="$h_x$", ylabel="$h_y$")
    ax.legend(loc="upper right", fontsize=8)
    return ax

ax = top_view(HZ_SLICES)
ax.set_title(f"L={L}: the wall at fixed $h_z$ (inside = z-pol)")
plt.show()
# ax.figure.savefig(os.path.join(FIGS, "phase3d_trivial_wall_topview_L4.png"), dpi=300, bbox_inches="tight")

Scan the cut height interactively:

In [ ]:
@interact(hz=FloatSlider(value=0.4, min=0.2, max=1.0, step=0.05, continuous_update=False))
def _scan(hz):
    ax = top_view([round(hz, 2)])
    ax.set_title(f"$h_z$ = {hz:.2f}")
    plt.show()

## §5 Prediction from the Hamiltonian alone: product-state mean field (no NQS)

Every edge in the same product state along a unit vector **n**. Per edge, ⟨A_v⟩ = n_x^(legs), ⟨B_p⟩ = n_z^4, so

* **3D TC:** e(n) = −(1/3)·n_x⁶ − n_z⁴ − h·n  (L³ six-leg stars and 3L³ plaquettes per 3L³ edges)
* **2D TC:** e(n) = −½·n_x⁴ − ½·n_z⁴ − h·n  (N/2 four-leg stars, N/2 plaquettes)

The stabilizer terms are *wells* on the sphere of directions: B_p a deep, stiff well at **z**, A_v a well at **x** (shallow
in 3D, equal to B_p in 2D), **nothing at y**. A first-order wall = the minimising **n** jumps, which needs two coexisting
minima. Mean field has no topological phase: near the origin its best product state is the flux-free z-state (3D) or
x/z (2D), standing in for the pocket. Positions of walls are reliable-ish; where they END is overestimated (no fluctuations).

Reading the tally: **3D** -> x|z and y|z walls only (x|y: a handful of grid crossings at one spot, jump ≤ 0.12).
**2D** -> the x|z wall plus x|y / y|z walls hugging the h_y axis at |h| ≲ 1.4: mean field's version of the 2D first-order
cap near h_y=1 (in reality topological|y-pol, h_y,c = 1). Nothing in 3D plays the x|y role.

In [ ]:
MF_MODELS = {"3D TC": lambda n: -n[:, 0]**6 / 3 - n[:, 2]**4,
             "2D TC": lambda n: -(n[:, 0]**4 + n[:, 2]**4) / 2}
_th, _ph = np.meshgrid(np.linspace(0, np.pi / 2, 121), np.linspace(0, np.pi / 2, 121), indexing="ij")
NDIR = np.stack([np.sin(_th) * np.cos(_ph), np.sin(_th) * np.sin(_ph), np.cos(_th)], -1).reshape(-1, 3)

def mf_plane(model, fixed, value, g=np.arange(0, 2.2, 0.02)):
    # MF walls on the plane h[fixed] = value: [(a, b, |dn|, "p|q")], (a, b) = the two free field axes in x,y,z order;
    # "p|q" = the two components of n that trade places across the jump (x|y: n_z unchanged, etc.)
    e0 = MF_MODELS[model](NDIR)
    free = [k for k in range(3) if k != fixed]
    A, B = np.meshgrid(g, g, indexing="ij")
    H = np.zeros(A.shape + (3,)); H[..., fixed] = value; H[..., free[0]] = A; H[..., free[1]] = B
    Hf = H.reshape(-1, 3)
    N = NDIR[np.concatenate([(e0[None] - Hf[s:s + 1024] @ NDIR.T).argmin(1) for s in range(0, len(Hf), 1024)])]
    N = N.reshape(A.shape + (3,))
    out = []
    for d in (0, 1):
        D = np.diff(N, axis=d)
        dn = np.linalg.norm(D, axis=-1)
        for i, j in zip(*np.nonzero(dn > 0.08)):
            i2, j2 = (i + 1, j) if d == 0 else (i, j + 1)
            pair = "|".join(sorted("xyz"[k] for k in np.argsort(-np.abs(D[i, j]))[:2]))
            out.append(((A[i, j] + A[i2, j2]) / 2, (B[i, j] + B[i2, j2]) / 2, dn[i, j], pair))
    return out

# which components trade places across the MF walls? tally over horizontal slices filling the positive octant
for model in MF_MODELS:
    walls = {}
    for hz in np.arange(0.0, 1.45, 0.1):
        for a, b, dn, pair in mf_plane(model, 2, hz):
            walls.setdefault(pair, []).append((a, b, hz, dn))
    print(model)
    for pair, w in sorted(walls.items()):
        w = np.array(w)
        print(f"  {pair}: {len(w):4d} grid crossings  h_x {w[:, 0].min():.2f}-{w[:, 0].max():.2f}  "
              f"h_y {w[:, 1].min():.2f}-{w[:, 1].max():.2f}  h_z {w[:, 2].min():.1f}-{w[:, 2].max():.1f}  max jump {w[:, 3].max():.2f}")

In [ ]:
PAIR_MARK = {"x|z": "o", "y|z": "^", "x|y": "s"}
REF = {"3D TC": dict(line=[(1.0, 0.21), (1.2, 0.5), (1.5, 0.75), (1.8, 1.0)], end=(1.8, 1.0),
                     lab="QMC $L\\to\\infty$ (Linsel+ 2026): tip -> endpoint"),
       "2D TC": dict(line=[(0.3406, 0.3406), (0.418, 0.418)], end=(0.418, 0.418),
                     lab="QMC (Wu+ 2012): multicritical -> endpoint")}
fig, axs = plt.subplots(2, 2, figsize=(11, 10))
for r, model in enumerate(MF_MODELS):
    for c, (fixed, val, xl, yl, title) in enumerate(((2, 0.5, "$h_x$", "$h_y$", "top view, $h_z$ = 0.5"),
                                                     (1, 0.0, "$h_x$", "$h_z$", "side view, $h_y$ = 0"))):
        ax = axs[r, c]
        W = mf_plane(model, fixed, val)
        for pair, mk in PAIR_MARK.items():
            P = np.array([w[:3] for w in W if w[3] == pair]).reshape(-1, 3)
            if len(P):
                sc = ax.scatter(P[:, 0], P[:, 1], c=P[:, 2], cmap="Reds", vmin=0, vmax=1.3, s=10, marker=mk,
                                label=f"MF wall {pair}")
        if c == 1:
            ref = np.array(REF[model]["line"])
            ax.plot(*ref.T, color="k", lw=2.5, alpha=0.5, label=REF[model]["lab"])
            ax.plot(*REF[model]["end"], "o", mfc="w", mec="k", ms=7)
        ax.set(xlim=(0, 2.2), ylim=(0, 2.2), aspect="equal", xlabel=xl, ylabel=yl, title=f"{model} mean field: {title}")
        ax.legend(loc="upper right", fontsize=7)
fig.colorbar(sc, ax=axs, shrink=0.5, label="|jump of n|")
plt.show()
# fig.savefig(os.path.join(FIGS, "mf_walls_3d_vs_2d.png"), dpi=300, bbox_inches="tight")

## §6 The 2D toric code in the full field (literature), same style as §3

Fields in J=1 units (Dusuel et al. and Vidal et al. use J=½ -> doubled here). **Published anchors:** axis 2nd-order points
h_c=0.328 (3D-Ising, TFIM-dual); multicritical point M=(0.3406, 0, 0.3406) and the first-order x|z line M->K, K=(0.418, 0, 0.418)
(Vidal-Dusuel-Schmidt 2009, Wu-Deng-Prokof'ev 2012); h_y axis first order at exactly 1 (self-dual, Vidal et al. 2009);
full-sphere boundary (Dusuel-Kamfor-Orús-Schmidt-Vidal PRL 106, 107203 (2011)): 2nd-order faces around the h_x/h_z axes,
a first-order cap near h_y, meeting (green) at ≈(0, 0.98, 0.22) and (0.22, 0.98, 0); multicritical line h_x=h_z=0.34 up to
h_y≈0.92. **Schematic (not published):** the pocket surface between anchors (thin-plate RBF through them), and the x|z wall
for h_y>0 — drawn shrinking to zero where the multicritical line meets the cap (only its h_y=0 edge is known; duality×K pins
it to the h_x=h_z plane; it must close before the y-polarized regime).

**Names:** topological = deconfined (e and m gapped, free). Large h_z: charges condense = **Higgs** (z-pol, B_p≈1).
Large h_x: fluxes condense, charges confined by string tension = **confined** (x-pol, A_v≈1). Higgs and confined are one
trivial phase (Fradkin-Shenker): the first-order line between them ends. y-pol (σ^y makes both e and m) is trivial too.

In [ ]:
def star_surface(P, orders, n=60, smoothing=1e-3):
    # star-shaped surface through boundary points P over the positive octant + 2nd-order indicator
    th, ph = np.meshgrid(np.linspace(0, np.pi / 2, n), np.linspace(0, np.pi / 2, n))
    U = np.stack([np.sin(th) * np.cos(ph), np.sin(th) * np.sin(ph), np.cos(th)], -1).reshape(-1, 3)
    R = sphere_rbf(P, np.linalg.norm(P, axis=1), smoothing)(U)
    second = sphere_rbf(P, (np.asarray(orders) == 2).astype(float), 1e-2)(U)
    X, Y, Z = (U * R[:, None]).T.reshape(3, n, n)
    return X, Y, Z, np.clip(second.reshape(n, n), 0, 1)

M2, K2, HY_TOP = 0.3406, 0.418, 0.92
pts, ords = [], []
for hy, axis in ((0.0, 0.328), (0.5, 0.30), (0.9, 0.24)):          # axis h_c(h_y): 0.328 -> 0.22 at h_y=0.98 (interpolated)
    diag = M2 if hy == 0 else 0.34
    for t in np.linspace(0, 1, 6):                                   # square-ish cross-section: axis point -> diagonal corner
        a, b = axis + t * (diag - axis), t * diag
        pts += [(a, hy, b), (b, hy, a)]; ords += [2, 2]
cap = [(0, 1.0, 0), (0.22, 0.98, 0), (0, 0.98, 0.22), (0.34, HY_TOP, 0.34), (0.15, 0.99, 0.15), (0.12, 0.995, 0), (0, 0.995, 0.12)]
pts += cap; ords += [1] * len(cap)
P2 = np.unique(np.round(np.array(pts, float), 6), axis=0)
O2 = [dict(zip(map(tuple, np.round(np.array(pts, float), 6)), ords))[tuple(p)] for p in P2]
X2, Y2, Z2, sec2 = star_surface(P2, O2)

hyw = np.linspace(0, HY_TOP, 25)
wK = M2 + (K2 - M2) * np.sqrt(np.clip(1 - (hyw / HY_TOP) ** 2, 0, 1))          # SKETCH: only h_y=0 is published
Mline = np.column_stack([np.full_like(hyw, M2), hyw, np.full_like(hyw, M2)])
Kline = np.column_stack([wK, hyw, wK])
V = np.vstack([Mline, Kline]); nW = len(hyw)
F = np.array([(k, k + 1, nW + k) for k in range(nW - 1)] + [(k + 1, nW + k + 1, nW + k) for k in range(nW - 1)])
green = np.array([(0.22, 0.98, 0), (0.34, HY_TOP, 0.34), (0, 0.98, 0.22)])

fig2 = go.Figure()
fig2.add_trace(go.Surface(x=X2, y=Y2, z=Z2, surfacecolor=sec2, cmin=0, cmax=1, showscale=False, opacity=0.55,
                          colorscale=[[0, "#f4b4b4"], [1, "#b4c8f4"]], name="pocket (schematic; red 1st, blue 2nd)",
                          showlegend=True, hoverinfo="skip"))
fig2.add_trace(go.Mesh3d(x=V[:, 0], y=V[:, 1], z=V[:, 2], i=F[:, 0], j=F[:, 1], k=F[:, 2], color=RED, opacity=0.6,
                         name="x|z wall (h_y=0 edge published; rest SKETCH)", showlegend=True, hoverinfo="skip"))
fig2.add_trace(go.Scatter3d(x=Mline[:, 0], y=Mline[:, 1], z=Mline[:, 2], mode="lines", line=dict(color="black", width=6),
                            name="multicritical line h_x=h_z (T analogue)"))
fig2.add_trace(go.Scatter3d(x=Kline[:, 0], y=Kline[:, 1], z=Kline[:, 2], mode="lines",
                            line=dict(color="#7f1d1d", width=6, dash="dot"), name="wall end (critical endpoints)"))
fig2.add_trace(go.Scatter3d(x=green[:, 0], y=green[:, 1], z=green[:, 2], mode="lines+markers",
                            line=dict(color="green", width=6), marker=dict(size=3), name="2nd-order faces meet 1st-order cap"))
anch = np.array([(0.328, 0, 0), (0, 0, 0.328), (M2, 0, M2), (K2, 0, K2), (0, 1, 0)])
fig2.add_trace(go.Scatter3d(x=anch[:, 0], y=anch[:, 1], z=anch[:, 2], mode="markers+text",
                            text=["0.328", "0.328", "M", "K", "h_y,c=1"], marker=dict(size=4, color="black"),
                            name="published anchors"))
fig2.update_layout(scene=dict(xaxis=dict(title="h_x", range=[0, 0.7]), yaxis=dict(title="h_y", range=[0, 1.2]),
                              zaxis=dict(title="h_z", range=[0, 0.7]), aspectmode="manual",
                              aspectratio=dict(x=0.7, y=1.2, z=0.7), camera=dict(eye=dict(x=1.7, y=1.3, z=0.9))),
                   scene_annotations=[dict(x=0.65, y=0.05, z=0.05, text="x-pol (confined)", showarrow=False),
                                      dict(x=0.05, y=0.05, z=0.65, text="z-pol (Higgs)", showarrow=False),
                                      dict(x=0.1, y=1.15, z=0.1, text="y-pol", showarrow=False)],
                   legend=dict(x=0, y=1), height=720, margin=dict(l=0, r=0, t=30, b=0),
                   title="2D toric code: pocket + the only trivial wall (x|z, on the self-dual plane)")
fig2.show()

## §7 2D vs 3D side by side at h_y = 0

Same structure (Fradkin–Shenker corner, a first-order Higgs|confined line leaving it and ending), different details:
2D is x↔z symmetric, both faces 2nd order, wall pinned on the diagonal and short; 3D has a first-order confinement face,
a flat Higgs face, and a long wall that, off this plane, bends around toward h_y (the z|y flank).

In [ ]:
fig, (a2, a3) = plt.subplots(1, 2, figsize=(11, 4.6))
line = np.array([(0, 0.328474), (0.3, 0.333), (M2, M2)])
a2.fill(*np.vstack([[0, 0], line, line[::-1, ::-1][1:], [0, 0]]).T, color="#ebe8f3", lw=0)
a2.plot(*line.T, color=BLUE); a2.plot(*line[:, ::-1].T, color=BLUE, label="2nd order (3D Ising)")
a2.plot([M2, K2], [M2, K2], color=RED, lw=2, label="1st order x|z wall")
a2.plot(K2, K2, "o", mfc="w", mec=RED, ms=7, label="critical endpoint K")
a2.plot([K2, 0.7], [K2, 0.7], ":", color="0.5", label="crossover")
a2.set(xlim=(0, 0.7), ylim=(0, 0.7), aspect="equal", xlabel="$h_x$", ylabel="$h_z$", title="2D TC, $h_y$=0 (literature)")
a2.legend(loc="lower right", fontsize=7)

pl = [r for r in rows if r["kind"] != "ycut" and r["hy"] == 0.0]
el = xyz(sorted((r for r in pl if r["kind"] == "electric" and r["topo"]), key=lambda r: r["hx"]))
mg = xyz(sorted((r for r in pl if r["kind"] == "magnetic" and r["topo"]), key=lambda r: -r["hz"]))
tl = x_lines[0][0]
a3.fill(*np.vstack([[0, 0], el[:, [0, 2]], mg[:, [0, 2]]]).T, color="#ebe8f3", lw=0)
a3.plot(el[:, 0], el[:, 2], "o-", color=BLUE, ms=4, label="NQS L=4: 2nd order")
a3.plot(*np.vstack([el[-1], mg])[:, [0, 2]].T, "s-", color=RED, ms=4, label="NQS L=4: 1st order")
a3.plot(tl[:-1, 0], tl[:-1, 2], "s--", color=RED, ms=4, label="NQS L=4: x|z wall")
q = np.array([(0, 0.197), (1.0, 0.21), (1.0, 0.0)]); qt = np.array(REF["3D TC"]["line"])
a3.plot(q[:2, 0], q[:2, 1], color=BLUE, lw=4, alpha=0.3); a3.plot(q[1:, 0], q[1:, 1], color=RED, lw=4, alpha=0.3)
a3.plot(*qt.T, color=RED, lw=4, alpha=0.3, label="QMC $L\\to\\infty$ (Linsel+ 2026)")
a3.plot(1.8, 1.0, "o", mfc="w", mec=RED, ms=7)
a3.set(xlim=(0, 2.0), ylim=(0, 1.1), xlabel="$h_x$", ylabel="$h_z$", title="3D TC, $h_y$=0")
a3.legend(loc="upper left", fontsize=7)
plt.tight_layout()
plt.show()
# fig.savefig(os.path.join(FIGS, "hy0_2d_vs_3d.png"), dpi=300, bbox_inches="tight")